# 07 - Autoencoder Anomaly Detection

The autoencoder is trained **only on normal recordings** (y == 0). It learns to reconstruct the normal pump sound. Anomalies are detected by reconstruction error: abnormal recordings are poorly reconstructed, producing high error.

**Label convention:** `0 = normal`, `1 = abnormal`.

Steps:
1. Load and preprocess data (identical to `05_cnn_classifier.ipynb`).
2. Split: train normal-only, validation, and a held-out test set with both classes.
3. Train a convolutional autoencoder on normal samples only.
4. Compute reconstruction errors and pick an anomaly threshold from the validation set.
5. Evaluate anomaly detection on the held-out test set.

In [ ]:
import numpy as np

X = np.load("../data/processed/X.npy")
y = np.load("../data/processed/y.npy")

print(X.shape)
print(y.shape)

In [ ]:
X = (X - X.min()) / (X.max() - X.min())
X = X[..., np.newaxis]

In [ ]:
from sklearn.model_selection import train_test_split

# Reproduce the same split as 05 so the test set is identical across notebooks
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

# Autoencoder trains on normal recordings only
X_normal = X_train[y_train == 0]
print("Normal training samples for AE:", X_normal.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# Split normal recordings into train (80%) and validation (20%)
X_ae_train, X_ae_val = train_test_split(
    X_normal,
    test_size=0.2,
    random_state=42
)

print("AE train:", X_ae_train.shape)
print("AE val:", X_ae_val.shape)

In [ ]:
# Pad the 313-wide spectrograms to 320 so MaxPool/UpSampling produce clean sizes.
# Reconstruction output is cropped back to (128, 313) before computing the error.
def pad_spectrograms(x):
    pad = np.zeros((x.shape[0], x.shape[1], 320, 1), dtype=x.dtype)
    pad[:, :, :x.shape[2], :] = x
    return pad

X_ae_train_p = pad_spectrograms(X_ae_train)
X_ae_val_p = pad_spectrograms(X_ae_val)

print("Padded AE train:", X_ae_train_p.shape)
print("Padded AE val:", X_ae_val_p.shape)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

input_shape = (128, 320, 1)

# --- Encoder ---
enc_in = layers.Input(shape=input_shape)

x = layers.Conv2D(32, kernel_size=3, activation='relu', padding='same')(enc_in)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, kernel_size=3, activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, kernel_size=3, activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(256, kernel_size=3, activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)

encoder = keras.Model(enc_in, x, name="encoder")
encoder.summary()

In [ ]:
# --- Decoder ---
dec_in = layers.Input(shape=encoder.output_shape[1:])

x = layers.Conv2DTranspose(128, kernel_size=3, strides=2, activation='relu', padding='same')(dec_in)
x = layers.BatchNormalization()(x)
x = layers.Conv2DTranspose(64, kernel_size=3, strides=2, activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2DTranspose(32, kernel_size=3, strides=2, activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv2D(1, kernel_size=3, activation='sigmoid', padding='same')(x)

decoder = keras.Model(dec_in, x, name="decoder")
decoder.summary()

In [ ]:
class AE(keras.Model):
    """Autoencoder that crops the reconstruction back to (128, 313)."""
    def __init__(self, encoder, decoder, width=313, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.width = width

    def call(self, inputs):
        z = self.encoder(inputs)
        recon = self.decoder(z)
        return recon

    def get_config(self):
        config = super().get_config()
        config.update({
            "encoder": keras.utils.serialize_keras_object(self.encoder),
            "decoder": keras.utils.serialize_keras_object(self.decoder),
            "width": self.width,
        })
        return config

    @classmethod
    def from_config(cls, config):
        config = dict(config)
        config["encoder"] = keras.utils.deserialize_keras_object(config["encoder"])
        config["decoder"] = keras.utils.deserialize_keras_object(config["decoder"])
        return cls(**config)

ae = AE(encoder, decoder)
ae.compile(optimizer='adam', loss='mse')
ae.build(input_shape=(None, *input_shape))
ae.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    save_best_only=True,
    filepath='../models/autoencoder.keras'
)

history = ae.fit(
    X_ae_train_p, X_ae_train_p,
    epochs=30,
    batch_size=32,
    validation_data=(X_ae_val_p, X_ae_val_p),
    callbacks=[early_stopping, checkpoint],
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Autoencoder training')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
def reconstruction_error(model, x, width=313):
    recon = model.predict(x, verbose=0)
    err = np.mean(np.square(x[:, :, :width, :] - recon[:, :, :width, :]), axis=(1, 2, 3))
    return err, recon

# Error on normal validation set (used to pick threshold)
val_err, _ = reconstruction_error(ae, X_ae_val_p)

threshold = np.percentile(val_err, 95)
print(f"Validation reconstruction error stats:")
print(f"  mean: {val_err.mean():.6f}")
print(f"  std:  {val_err.std():.6f}")
print(f"  95th percentile threshold: {threshold:.6f}")

In [ ]:
# Evaluate on the held-out test set (contains both normal and abnormal)
X_test_p = pad_spectrograms(X_test)
test_err, _ = reconstruction_error(ae, X_test_p)

y_pred_ae = (test_err > threshold).astype(int)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print(classification_report(
    y_test,
    y_pred_ae,
    target_names=["normal", "abnormal"],
    digits=4
))
print(f"Accuracy:  {accuracy_score(y_test, y_pred_ae):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_ae):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_ae):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_ae):.4f}")
print(f"ROC AUC (error as score): {roc_auc_score(y_test, test_err):.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, mask, label in zip(axes, [y_test == 0, y_test == 1], ["normal", "abnormal"]):
    ax.hist(test_err[mask], bins=50, alpha=0.6, label=label)
    ax.axvline(threshold, color="red", linestyle="--", label=f"threshold = {threshold:.4f}")
    ax.set_xlabel("Reconstruction error (MSE)")
    ax.set_ylabel("Count")
    ax.set_title(f"Test reconstruction error - {label}")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize reconstruction quality on a normal and an abnormal sample
from sklearn.metrics import roc_curve, auc

idx_normal = np.where(y_test == 0)[0][0]
idx_abnormal = np.where(y_test == 1)[0][0]

for idx, title in [(idx_normal, "normal"), (idx_abnormal, "abnormal")]:
    sample = X_test_p[idx:idx + 1]
    recon = ae.predict(sample, verbose=0)[0, :, :313, 0]
    orig = sample[0, :, :313, 0]
    err = np.mean(np.square(orig - recon))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, data, t in zip(axes, [orig, recon, np.abs(orig - recon)], ["Original", "Reconstruction", "|Error|"]):
        im = ax.imshow(data, aspect="auto", origin="lower", cmap="magma")
        ax.set_title(f"{title} - {t} (MSE={err:.5f})")
        fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()